# 📚 Naive RAG do Zero - Tutorial Completo

**Curso AI Engineers - Naive RAG**

Neste tutorial, vamos implementar uma arquitetura **Naive RAG (Retrieval-Augmented Generation)** completa seguindo o pipeline:

```
Ingestão → Embeddings → Recuperação → Geração
```

## 🎯 O que vamos construir?

Um sistema que:
1. **Carrega** um documento PDF
2. **Fragmenta** o texto em chunks
3. **Converte** em embeddings vetoriais
4. **Armazena** em um banco vetorial (ChromaDB)
5. **Recupera** contexto relevante via similaridade
6. **Gera** respostas usando LLM

## 🔧 Implementação Direta
Este tutorial usa **bibliotecas essenciais** sem frameworks abstratos como LangChain!

## 📦 Parte 1: Instalação de Dependências

Vamos usar apenas as bibliotecas essenciais:

In [ ]:
# Instalar dependências
!pip install -q openai pypdf chromadb

## 🔑 Parte 2: Configuração da API OpenAI

Configure sua chave de API da OpenAI:

In [ ]:
import os

# Configurar chave da API OpenAI
# Você pode obter sua chave em: https://platform.openai.com/api-keys
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = "<INSIRA_API_KEY>"

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

## 📄 Parte 3: Ingestão de Dados

### Suporte a Múltiplas Fontes

O sistema suporta:
- ✅ PDFs
- ✅ Arquivos de texto (.txt)
- ✅ Markdown (.md)

### Limpeza Básica
Remoção de ruídos e formatação redundante para garantir qualidade.

In [ ]:
from pypdf import PdfReader
import re

def load_document(file_path: str) -> str:
    """
    Carrega um documento e retorna seu conteúdo como texto.
    
    Args:
        file_path: Caminho para o arquivo (PDF ou TXT)
    
    Returns:
        Texto extraído do documento
    """
    if file_path.endswith('.pdf'):
        # Carregar PDF
        reader = PdfReader(file_path)
        content = ""
        for page in reader.pages:
            content += page.extract_text() + "\n\n"
    else:
        # Carregar arquivo de texto
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
    
    # Limpeza básica: remover espaços extras e linhas vazias múltiplas
    content = re.sub(r'\n\s*\n', '\n\n', content)
    content = content.strip()
    
    return content

# Para este exemplo, vamos criar um documento de teste
test_content = """
# Inteligência Artificial e Machine Learning

## Introdução
A inteligência artificial (IA) é um campo da ciência da computação que se concentra na criação de sistemas capazes de realizar tarefas que normalmente requerem inteligência humana.

## Machine Learning
Machine Learning é um subcampo da IA que permite que computadores aprendam sem serem explicitamente programados. Os principais tipos incluem:

1. **Aprendizado Supervisionado**: O modelo aprende com dados rotulados
2. **Aprendizado Não Supervisionado**: O modelo encontra padrões em dados não rotulados
3. **Aprendizado por Reforço**: O modelo aprende através de tentativa e erro

## Deep Learning
Deep Learning utiliza redes neurais profundas para processar dados complexos. É particularmente eficaz em:
- Reconhecimento de imagens
- Processamento de linguagem natural
- Análise de vídeo

## RAG (Retrieval-Augmented Generation)
RAG é uma técnica que combina recuperação de informações com geração de texto. O sistema:
1. Recupera documentos relevantes de uma base de conhecimento
2. Usa esses documentos como contexto para gerar respostas precisas
3. Reduz alucinações do modelo de linguagem

## Aplicações Práticas
A IA tem aplicações em diversos setores:
- **Saúde**: Diagnóstico médico assistido
- **Finanças**: Detecção de fraudes
- **Educação**: Sistemas de tutoria inteligentes
- **Varejo**: Sistemas de recomendação

## Embeddings e Vetorização
Embeddings são representações vetoriais de texto que capturam significado semântico. Textos similares têm vetores próximos no espaço multidimensional.

## Bancos de Dados Vetoriais
Bancos vetoriais como FAISS, Chroma e Pinecone permitem busca eficiente por similaridade em milhões de vetores.
"""

# Salvar documento de teste
with open('documento_teste.txt', 'w', encoding='utf-8') as f:
    f.write(test_content)

# Carregar documento
raw_text = load_document('documento_teste.txt')
print(f"📄 Documento carregado!")
print(f"📊 Caracteres: {len(raw_text)}")
print(f"\n📝 Preview (primeiros 300 caracteres):\n{raw_text[:300]}...")

## ✂️ Parte 4: Fragmentação de Documentos (Chunking)

### 🎯 Por que Fragmentar?

Como vimos na aula, o **chunking é engenharia** - a divisão correta do documento é o desafio central no RAG.

### ⚙️ Parâmetros Principais:

- **Chunk Size**: Tamanho do bloco (500-1000 tokens recomendado)
- **Overlap**: Sobreposição entre chunks (ajuda a manter contexto)

### ⚠️ Riscos:
- **Fragmentação inadequada** → Quebra semântica e reduz precisão
- **Chunks muito pequenos** → Perda de contexto
- **Chunks muito grandes** → Informação irrelevante no contexto

In [ ]:
from typing import List

def chunk_text_recursive(text: str, chunk_size: int = 500, chunk_overlap: int = 50) -> List[str]:
    """
    Fragmenta o texto em chunks usando estratégia recursiva.

    Estratégia:
    1. Tenta dividir usando separadores mais "semânticos" primeiro.
    2. Se não conseguir dividir, tenta separadores mais granulares.
    3. Garante que o chunk_size seja respeitado.
    4. Mantém sobreposição (overlap) entre chunks.
    """

    # 🛑 Caso base:
    # Se o texto já for menor que o tamanho máximo permitido,
    # não é necessário fragmentar.
    if len(text) <= chunk_size:
        return [text]

    # 🎯 Separadores ordenados por prioridade semântica.
    # A ideia é preservar o máximo de significado possível.
    separators = ["\n\n", "\n", ". ", " ", ""]

    def split_text(text: str, separators: List[str]) -> List[str]:
        """
        Divide o texto usando o primeiro separador que conseguir
        fragmentar o conteúdo.

        Se não conseguir dividir com o separador atual,
        chama a si mesma (recursão) com o próximo separador.
        """

        # 🛑 Caso base da recursão:
        # Se acabarem os separadores, retorna o texto inteiro.
        if not separators:
            return [text]

        separator = separators[0]      # Primeiro separador da lista
        remaining = separators[1:]     # Restante dos separadores

        # 🔎 Tenta dividir usando o separador atual
        if separator:
            splits = text.split(separator)
        else:
            # 🚨 Último recurso:
            # Divide caractere por caractere.
            # Garante que SEMPRE será possível fragmentar.
            splits = list(text)

        # ⚠️ Se não conseguiu dividir (apenas 1 elemento),
        # tenta novamente com o próximo separador.
        if len(splits) == 1:
            return split_text(text, remaining)

        # ✅ Conseguiu dividir
        return splits

    # 🔹 Primeira divisão do texto
    splits = split_text(text, separators)

    print("\n\nSplits", splits, "\n\n")

    chunks = []
    current_chunk = ""

    # 📦 Construção dos chunks respeitando chunk_size
    for i, split in enumerate(splits):

        # 🔎 Cria candidato a chunk:
        # Se já houver conteúdo acumulado, adiciona um espaço
        # antes do novo split para manter legibilidade.
        candidate = (current_chunk + " " + split).strip() if current_chunk else split

        print(f"Split: {i}, the candidate looks like:\n<candidate>{candidate}<candidate>\nIt's length is: {len(candidate)} \n\n")

        # ✅ Se ainda cabe dentro do limite, continua acumulando
        if len(candidate) <= chunk_size:
            current_chunk = candidate
        else:
            # 🚀 Se ultrapassar o limite:

            # 1️⃣ Salva o chunk atual
            if current_chunk:
                chunks.append(current_chunk)
                print(f"The chunk added was:\n<chunk>{current_chunk}<chunk>")

            # 2️⃣ Inicia novo chunk aplicando overlap
            if chunk_overlap > 0 and current_chunk:
                # Pega os últimos caracteres do chunk anterior
                overlap_text = current_chunk[-chunk_overlap:]

                # Novo chunk começa com o overlap + novo split
                current_chunk = (overlap_text + " " + split).strip()
            else:
                # Se não houver overlap
                current_chunk = split
              
            print(f"\nAnd the current chunk looks like:\n<current>{current_chunk}<current>\n\n")

    # 📌 Adiciona o último chunk pendente
    if current_chunk:
        chunks.append(current_chunk)

    return chunks

# Fragmentar documento
chunks = chunk_text_recursive(raw_text, chunk_size=500, chunk_overlap=50)

print(f"✂️ Documento fragmentado em {len(chunks)} chunks\n")
print("📋 Exemplos de chunks:\n")
for i, chunk in enumerate(chunks[:3], 1):
    print(f"--- Chunk {i} ({len(chunk)} caracteres) ---")
    print(chunk)
    print()

## 🧠 Parte 5: Incorporações (Embeddings)

### Princípio Central
Converte texto não estruturado em **vetores matemáticos** em um espaço de alta dimensão.

### Captura Semântica
Vetores próximos no espaço representam significados semelhantes, permitindo correspondência difusa.

### ⚠️ Consistência do Modelo
**CRÍTICO**: O texto da consulta e os documentos indexados devem usar **exatamente o mesmo modelo** de embedding!

### 🎯 Embeddings definem o teto
Como visto na aula: a eficácia do modelo de Embedding determina diretamente o **limite máximo da qualidade** da recuperação semântica.

In [ ]:
from openai import OpenAI
from typing import List

# Inicializar cliente OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

def get_embedding(text: str, model: str = "text-embedding-3-small") -> List[float]:
    """
    Gera embedding para um texto usando a API OpenAI.
    
    Args:
        text: Texto para gerar embedding
        model: Modelo de embedding da OpenAI
    
    Returns:
        Lista de floats representando o vetor de embedding
    """
    # Limpar texto
    text = text.replace("\n", " ").strip()
    
    # Chamar API
    response = client.embeddings.create(
        input=[text],
        model=model
    )
    
    return response.data[0].embedding

def get_embeddings_batch(texts: List[str], model: str = "text-embedding-3-small") -> List[List[float]]:
    """
    Gera embeddings para múltiplos textos em batch.
    
    Args:
        texts: Lista de textos
        model: Modelo de embedding da OpenAI
    
    Returns:
        Lista de vetores de embedding
    """
    # Limpar textos
    cleaned_texts = [text.replace("\n", " ").strip() for text in texts]
    
    # Chamar API em batch
    response = client.embeddings.create(
        input=cleaned_texts,
        model=model
    )
    
    return [item.embedding for item in response.data]

# Testar o modelo com um exemplo
test_text = "O que é inteligência artificial?"
test_embedding = get_embedding(test_text)

print(f"🧠 Modelo de Embeddings: text-embedding-3-small")
print(f"📐 Dimensões do vetor: {len(test_embedding)}")
print(f"\n🔢 Primeiros 10 valores do vetor:")
print([round(v, 4) for v in test_embedding[:10]])

## 🗄️ Parte 6: Armazenamento Vetorial (ChromaDB)

### Ecossistema de Ferramentas

**Opções Leve/Open Source:**
- ✅ **ChromaDB** (Local): Fácil implantação, ideal para protótipos
- ✅ **FAISS** (Facebook AI): Local, ideal para prototipagem

**Serviços Gerenciados/Nuvem:**
- ☁️ **Pinecone**: Escalabilidade horizontal massiva
- ☁️ **Weaviate**: Alta disponibilidade

### Mecanismo de Indexação
Algoritmos **HNSW** ou **IVF** para recuperação rápida sublinear.

### Métrica de Similaridade
**Cosseno** ou **Distância Euclidiana** para vizinhos espaciais.

In [ ]:
import chromadb
from chromadb.config import Settings
from typing import List, Dict, Any

# Inicializar ChromaDB
# Usando modo persistente (salva em disco)
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# Criar ou obter coleção
collection_name = "naive_rag_collection"

# Deletar coleção se já existir (para garantir começar do zero)
try:
    chroma_client.delete_collection(name=collection_name)
except:
    pass

# Criar nova coleção
collection = chroma_client.create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"}  # Usar similaridade de cosseno
)

print(f"🗄️ Coleção ChromaDB criada: {collection_name}")

# Gerar embeddings para todos os chunks
print(f"\n🧠 Gerando embeddings para {len(chunks)} chunks...")
chunk_embeddings = get_embeddings_batch(chunks)
print(f"✓ Embeddings gerados!")

# Adicionar documentos ao ChromaDB
print(f"\n💾 Adicionando documentos ao ChromaDB...")

# Preparar dados
ids = [f"chunk_{i}" for i in range(len(chunks))]
metadatas = [{"chunk_id": i, "source": "documento_teste.txt"} for i in range(len(chunks))]

# Adicionar à coleção
collection.add(
    ids=ids,
    embeddings=chunk_embeddings,
    documents=chunks,
    metadatas=metadatas
)

print(f"✅ {len(chunks)} documentos adicionados ao ChromaDB!")
print(f"📊 Total de documentos na coleção: {collection.count()}")

## 🔍 Parte 7: Recuperação Semântica (Retrieval)

### Similaridade de Cosseno
Calcula a distância geométrica entre o **vetor de consulta** e os **vetores do documento**.

### Fórmula
$$\text{similaridade} = \frac{A \cdot B}{||A|| \times ||B||}$$

### Estratégia Top-k
Recupera os **3-5 fragmentos mais relevantes** como contexto de referência.

### 🎯 Núcleo da Qualidade
Como visto na aula: **A qualidade da entrada e a precisão da indexação determinam o efeito final da recuperação.**

### ⚡ Eficiência na Recuperação
Correspondência em **milissegundos** através de indexação em bancos vetoriais.

In [ ]:
from typing import Tuple, List, Dict

def retrieve_context(query: str, collection, k: int = 3) -> Tuple[str, List[Dict]]:
    """
    Recupera os top-k documentos mais similares à consulta.
    
    Args:
        query: Pergunta do usuário
        collection: Coleção ChromaDB
        k: Número de documentos a recuperar
    
    Returns:
        Tupla (contexto_concatenado, lista_de_resultados)
    """
    # Gerar embedding da consulta
    query_embedding = get_embedding(query)
    
    # Buscar documentos similares no ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )
    
    # Extrair dados dos resultados
    retrieved_docs = []
    context_parts = []
    
    # ChromaDB retorna listas de listas, então pegamos o primeiro conjunto
    documents = results['documents'][0]
    distances = results['distances'][0]
    metadatas = results['metadatas'][0]
    ids = results['ids'][0]
    
    for i, doc in enumerate(documents):
        # Converter distância em score de similaridade (1 - distância)
        similarity_score = 1 - distances[i]
        
        retrieved_docs.append({
            'text': doc,
            'score': similarity_score,
            'metadata': metadatas[i],
            'id': ids[i]
        })
        context_parts.append(doc)
    
    # Concatenar contexto
    context = "\n\n".join(context_parts)
    
    return context, retrieved_docs

# Testar recuperação
test_query = "O que é RAG e como funciona?"
print(f"❓ Consulta: {test_query}\n")

context, retrieved_docs = retrieve_context(test_query, collection, k=3)

print(f"📚 Documentos recuperados ({len(retrieved_docs)}):\n")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"--- Documento {i} (score: {doc['score']:.4f}) ---")
    print(doc['text'][:200] + "..." if len(doc['text']) > 200 else doc['text'])
    print()

## 🤖 Parte 8: Geração de Conteúdo (Generation)

### Modelos de Prompt
Integração do contexto recuperado com a pergunta original do usuário.

### Lógica Central
O Prompt é a **cola** que conecta o conteúdo recuperado às capacidades do LLM.

### Restrições Comportamentais
Uso de **System Message** para garantir que o modelo responda apenas com base em fatos recuperados.

### Estrutura Típica
```
[Instrução da Tarefa] + [Contexto Recuperado] + [Consulta do Usuário]
```

In [ ]:
def generate_response(query: str, context: str, model: str = "gpt-4o-mini") -> str:
    """
    Gera uma resposta usando o LLM com o contexto recuperado.
    
    Args:
        query: Pergunta do usuário
        context: Contexto recuperado do vector store
        model: Nome do modelo OpenAI
    
    Returns:
        Resposta gerada pelo LLM
    """
    # Construir mensagens
    messages = [
        {
            "role": "system",
            "content": """Você é um assistente especializado que responde perguntas com base no contexto fornecido.

REGRAS IMPORTANTES:
- Use APENAS as informações do contexto fornecido para responder
- Se a resposta não estiver no contexto, diga "Não tenho informações suficientes para responder"
- Seja preciso e objetivo
- Cite partes relevantes do contexto quando apropriado"""
        },
        {
            "role": "user",
            "content": f"""Contexto:
{context}

Pergunta: {query}

Resposta:"""
        }
    ]
    
    # Chamar API OpenAI
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0
    )
    
    return response.choices[0].message.content

# Testar geração de resposta
query = "O que é RAG e como funciona?"
response = generate_response(query, context)

print(f"❓ Pergunta: {query}\n")
print(f"🤖 Resposta:\n{response}")

## 🚀 Parte 9: Sistema RAG Completo

Agora vamos juntar tudo em uma função única que executa o pipeline completo!

In [ ]:
def naive_rag_pipeline(query: str, collection, k: int = 3, verbose: bool = True) -> str:
    """
    Pipeline completo de Naive RAG.
    
    Args:
        query: Pergunta do usuário
        collection: Coleção ChromaDB
        k: Número de documentos a recuperar
        verbose: Se True, imprime informações de debug
    
    Returns:
        Resposta final gerada
    """
    if verbose:
        print("="*70)
        print("🔄 INICIANDO PIPELINE NAIVE RAG")
        print("="*70)
    
    # Etapa 1: Vetorização da consulta
    if verbose:
        print("\n📝 Etapa 1: Vetorização da Consulta")
        print(f"   Query: {query}")
    
    query_embedding = get_embedding(query)
    
    if verbose:
        print(f"   ✓ Embedding gerado: {len(query_embedding)} dimensões")
    
    # Etapa 2: Recuperação semântica
    if verbose:
        print(f"\n🔍 Etapa 2: Recuperação Semântica (Top-{k})")
    
    context, docs = retrieve_context(query, collection, k=k)
    
    if verbose:
        print(f"   ✓ {len(docs)} documentos recuperados")
        for i, doc in enumerate(docs, 1):
            preview = doc['text'][:100].replace('\n', ' ')
            print(f"   [{i}] Score: {doc['score']:.4f} | {preview}...")
    
    # Etapa 3: Montagem do prompt
    if verbose:
        print("\n🧩 Etapa 3: Montagem do Prompt")
        print(f"   ✓ Contexto: {len(context)} caracteres")
    
    # Etapa 4: Geração da resposta
    if verbose:
        print("\n🤖 Etapa 4: Geração (LLM)")
    
    response = generate_response(query, context)
    
    if verbose:
        print(f"   ✓ Resposta gerada: {len(response)} caracteres")
        print("\n" + "="*70)
        print("✅ PIPELINE CONCLUÍDO")
        print("="*70)
        print(f"\n📤 RESPOSTA FINAL:\n")
        print(response)
        print("\n" + "="*70)
    
    return response

# Executar pipeline completo
pergunta = "Quais são os tipos de Machine Learning?"
resposta = naive_rag_pipeline(pergunta, collection, k=3)

## 🧪 Parte 10: Testando o Sistema

Vamos fazer várias perguntas para testar nosso sistema RAG!

In [ ]:
# Lista de perguntas para testar
perguntas_teste = [
    "O que é Deep Learning?",
    "Quais são as aplicações práticas da IA?",
    "Como o RAG reduz alucinações?",
    "O que é aprendizado por reforço?",
    "O que são embeddings?"
]

print("🧪 TESTANDO O SISTEMA RAG\n")

for i, pergunta in enumerate(perguntas_teste, 1):
    print(f"\n{'='*70}")
    print(f"TESTE {i}/{len(perguntas_teste)}")
    print(f"{'='*70}")
    
    resposta = naive_rag_pipeline(pergunta, collection, k=2, verbose=False)
    
    print(f"\n❓ Pergunta: {pergunta}")
    print(f"\n🤖 Resposta:\n{resposta}")
    print()

## 🎓 Próximos Passos

### 📚 Para Aprofundar:

1. **Experimentar com seus próprios documentos**
   - PDFs técnicos
   - Documentação de produtos
   - Base de conhecimento interna

2. **Otimizar parâmetros**
   - Chunk size (500 vs 1000 vs 2000)
   - Overlap (0 vs 50 vs 100)
   - Top-k (3 vs 5 vs 10)

3. **Testar diferentes modelos**
   - Embeddings: text-embedding-3-small vs text-embedding-3-large
   - LLM: gpt-4o-mini vs gpt-4o vs gpt-4-turbo

4. **Implementar melhorias**
   - Adicionar metadados aos chunks
   - Implementar filtros de recuperação
   - Criar sistema de avaliação de respostas

## 🙏 Conclusão

Parabéns! Você construiu um sistema **Naive RAG completo** usando ChromaDB! 🎉

### O que você aprendeu:
- ✅ Implementar chunking recursivo
- ✅ Gerar embeddings com OpenAI API
- ✅ Usar ChromaDB para armazenamento vetorial
- ✅ Recuperar documentos por similaridade
- ✅ Integrar tudo em um pipeline RAG completo

### Vantagens desta implementação:
- 🎯 **Simplicidade**: Código direto e compreensível
- 🔧 **Profissional**: ChromaDB é usado em produção
- 💾 **Persistência**: Dados salvos automaticamente
- 📚 **Educacional**: Perfeito para aprender os fundamentos

**Lembre-se**: A prática leva à perfeição. Continue experimentando!

---

*Instrutor: Equipe AI Engineering Academy*  
*Dia 2 do Curso*